# Bare `como` Dataset Expansion Experiment v2

This notebook is isolated from the main 01 evaluation workflow. It runs only the stricter bare-`como` experiment over a configurable bronze/raw corpus and writes to the experiment namespace:

```text
data/gold/experiments/bare_como_curated_ground_v2/
outputs/experiments/bare_como_curated_ground_v2/
```

It does not write `data/gold/evaluation` or any spec-0006 output.

Pattern:

```text
CURATED_QUALITY_GROUND como VEHICLE
```

Excluded connector forms:

```text
como um / uma / uns / umas
```

v2 cleanup:
- keeps bare one-token vehicles such as `claro como água`;
- strips definite articles in `como o/a/os/as NOUN`, using the noun as `vehicle_head_clean`;
- rejects clause starts and pronoun/function-word heads such as `se`, `é`, `eu`, `ele`, `sempre`;
- writes full candidates, accepted candidates, counts, review sample, and quality notes.


In [1]:
import os
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src/tal_qual").exists():
            return candidate
        if (candidate / "work/src/tal_qual").exists():
            return candidate / "work"
    raise RuntimeError(f"Could not find repository root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

REPO_ROOT


PosixPath('/home/jovyan/work')

In [2]:
import tempfile
import zipfile

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

SPARK_MASTER = os.environ.get("TAL_QUAL_SPARK_MASTER", "local[4]")
SPARK_PARALLELISM = os.environ.get("TAL_QUAL_SPARK_PARALLELISM", "4")
SPARK_SHUFFLE_PARTITIONS = os.environ.get("TAL_QUAL_SPARK_SHUFFLE_PARTITIONS", "4")
SPARK_DRIVER_MEMORY = os.environ.get("TAL_QUAL_SPARK_DRIVER_MEMORY", "4g")

spark = (
    SparkSession.builder
    .master(SPARK_MASTER)
    .appName("tal-qual-bare-como-expansion-v2")
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)
    .config("spark.default.parallelism", SPARK_PARALLELISM)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTITIONS)
    .config("spark.sql.files.maxPartitionBytes", str(128 * 1024 * 1024))
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

package_zip = Path(tempfile.gettempdir()) / "tal_qual_src.zip"
with zipfile.ZipFile(package_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted((SRC_DIR / "tal_qual").rglob("*.py")):
        archive.write(file_path, file_path.relative_to(SRC_DIR))
spark.sparkContext.addPyFile(str(package_zip))

SPARK_MASTER, spark.version


('local[4]', '4.1.1')

In [3]:
from tal_qual.bronze import BRONZE_OUTPUT_PATH, RAW_CORPUS_INPUT, load_or_build_bronze_dataframe
from tal_qual.dataset_expansion_experiments import (
    accepted_candidates_dataframe,
    accepted_review_sample_dataframe,
    candidate_counts_dataframe,
    experiment_dataset_path,
    experiment_output_path,
    ground_vehicle_counts_dataframe,
    prefilter_dataset_expansion_bronze_dataframe,
    prepare_dataset_expansion_dataframe,
    review_sample_dataframe,
    vehicle_counts_dataframe,
    write_dataset_expansion_artifacts,
)


## Load Bronze

By default this uses the normal one-shard bronze path. For a full-corpus run, set the same variables used by the 01 full notebook before starting Docker/Jupyter:

```bash
TAL_QUAL_RAW_CORPUS_INPUT=data/raw/brwac-clean-*.txt.gz
TAL_QUAL_BRONZE_PATH=data/bronze/brwac_segments_full
```


In [4]:
bronze_path = REPO_ROOT / os.environ.get("TAL_QUAL_BRONZE_PATH", str(BRONZE_OUTPUT_PATH))
raw_path = REPO_ROOT / os.environ.get("TAL_QUAL_RAW_CORPUS_INPUT", str(RAW_CORPUS_INPUT))

bronze_df = load_or_build_bronze_dataframe(spark, raw_path, bronze_path).persist()
bronze_rows = bronze_df.count()
print(f"Raw path: {raw_path}")
print(f"Bronze path: {bronze_path}")
print(f"Bronze rows: {bronze_rows:,}")
bronze_df.printSchema()


Raw path: /home/jovyan/work/data/raw/brwac-clean-1.txt.gz
Bronze path: /home/jovyan/work/data/bronze/brwac_segments
Bronze rows: 4,673,057
root
 |-- source_file: string (nullable = true)
 |-- original_line_id: long (nullable = true)
 |-- segment_id: integer (nullable = true)
 |-- text_original: string (nullable = true)
 |-- text_normalized: string (nullable = true)
 |-- match_text: string (nullable = true)



## Extract Bare `como` v2


In [5]:
EXPERIMENT_SLUG = "bare_como_curated_ground_v2"

prefiltered_df = prefilter_dataset_expansion_bronze_dataframe(bronze_df, EXPERIMENT_SLUG).persist()
prefiltered_rows = prefiltered_df.count()
print(f"Prefiltered bronze rows: {prefiltered_rows:,}")

candidates_df = prepare_dataset_expansion_dataframe(prefiltered_df, EXPERIMENT_SLUG).persist()
candidate_rows = candidates_df.count()
print(f"Candidate rows: {candidate_rows:,}")

accepted_df = accepted_candidates_dataframe(candidates_df).persist()
accepted_rows = accepted_df.count()
accepted_rate = accepted_rows / candidate_rows if candidate_rows else 0.0
print(f"Accepted rows: {accepted_rows:,} ({accepted_rate:.1%})")

accepted_path = experiment_dataset_path(EXPERIMENT_SLUG).parent / "accepted_candidates"
write_dataset_expansion_artifacts(candidates_df, EXPERIMENT_SLUG)
accepted_df.write.mode("overwrite").parquet(str(accepted_path))

(
    accepted_review_sample_dataframe(candidates_df, limit=500)
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(REPO_ROOT / experiment_output_path(EXPERIMENT_SLUG) / "accepted_review_sample.csv"))
)
print(f"Full candidate dataset: {experiment_dataset_path(EXPERIMENT_SLUG)}")
print(f"Accepted candidate dataset: {accepted_path}")
print(f"Review artifacts: {experiment_output_path(EXPERIMENT_SLUG)}")


Prefiltered bronze rows: 1,170


/usr/local/spark/python/pyspark/sql/udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


Candidate rows: 1,184
Accepted rows: 772 (65.2%)
Full candidate dataset: data/gold/experiments/bare_como_curated_ground_v2/candidates
Accepted candidate dataset: data/gold/experiments/bare_como_curated_ground_v2/accepted_candidates
Review artifacts: outputs/experiments/bare_como_curated_ground_v2


## Decision Tables


In [6]:
candidate_counts_dataframe(candidates_df).show(truncate=False)

ground_vehicle_counts_dataframe(accepted_df).show(40, truncate=False)

vehicle_counts_dataframe(accepted_df).show(40, truncate=False)


+---------------------------+-------------+-----+
|pattern_type               |quality_label|count|
+---------------------------+-------------+-----+
|bare_como_curated_ground_v2|trimmed      |408  |
|bare_como_curated_ground_v2|reject       |385  |
|bare_como_curated_ground_v2|keep         |364  |
|bare_como_curated_ground_v2|review       |27   |
+---------------------------+-------------+-----+

+---------------------------+------------+------------------+-----+
|pattern_type               |ground_lemma|vehicle_head_clean|count|
+---------------------------+------------+------------------+-----+
|bare_como_curated_ground_v2|claro       |água              |10   |
|bare_como_curated_ground_v2|escuro      |breu              |7    |
|bare_como_curated_ground_v2|claro       |luz               |6    |
|bare_como_curated_ground_v2|forte       |vermelho          |5    |
|bare_como_curated_ground_v2|pequeno     |grandes           |4    |
|bare_como_curated_ground_v2|pesado      |chumbo       

## Accepted Review Sample


In [7]:
accepted_review_sample_dataframe(candidates_df, limit=200).show(80, truncate=120)


+----------------------------------------+---------------------------+-----------+------------+--------------+------------------------------------------------------------------------------------------------------------------------+------------------------------------+---------------------+-------------+------------------------+------------+------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------+----------------+----------+
|                            candidate_id|               pattern_type|ground_text|ground_lemma|connector_text|                                                                                                        vehicle_text_raw|                  vehicle_text_clean|   vehicle_head_clean|quality_label|          quality_reason|needs_review|                                                                                                     candidate_

## Full Review Sample


In [8]:
review_sample_dataframe(candidates_df, limit=200).show(60, truncate=100)


+----------------------------------------+---------------------------+-----------+------------+--------------+----------------------------------------------------------------------------------------------------+----------------------------------+------------------+-------------+----------------------------+------------+----------------------------------------------------------------------------------------------------+------------------------------------------------------+----------------+----------+
|                            candidate_id|               pattern_type|ground_text|ground_lemma|connector_text|                                                                                    vehicle_text_raw|                vehicle_text_clean|vehicle_head_clean|quality_label|              quality_reason|needs_review|                                                                                 candidate_full_text|                                           source_file|original_line_id|

## Promotion Check


In [9]:
false_positive_reason_rows = (
    candidates_df
    .where(F.col("quality_label").isin("review", "reject"))
    .groupBy("quality_reason")
    .count()
    .orderBy(F.col("count").desc(), "quality_reason")
    .collect()
)

summary_rows = [
    {
        "experiment_slug": EXPERIMENT_SLUG,
        "prefiltered_rows": prefiltered_rows,
        "candidate_rows": candidate_rows,
        "accepted_rows": accepted_rows,
        "accepted_rate": accepted_rate,
        "distinct_grounds": accepted_df.select("ground_lemma").distinct().count(),
        "distinct_vehicle_heads": accepted_df.select("vehicle_head_clean").distinct().count(),
        "distinct_pairs": accepted_df.select("ground_lemma", "vehicle_head_clean").distinct().count(),
        "top_false_positive_reason": false_positive_reason_rows[0]["quality_reason"] if false_positive_reason_rows else "none",
        "promotion_recommendation": "review top pairs before promotion" if accepted_rate >= 0.60 else "needs another cleanup pass",
    }
]
summary_df = spark.createDataFrame(summary_rows)
summary_output = REPO_ROOT / experiment_output_path(EXPERIMENT_SLUG) / "promotion_summary.csv"
(
    summary_df
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(str(summary_output))
)
summary_df.show(truncate=False)


+-----------------+-------------+--------------+----------------+--------------+----------------------+---------------------------+----------------+---------------------------------+-------------------------+
|accepted_rate    |accepted_rows|candidate_rows|distinct_grounds|distinct_pairs|distinct_vehicle_heads|experiment_slug            |prefiltered_rows|promotion_recommendation         |top_false_positive_reason|
+-----------------+-------------+--------------+----------------+--------------+----------------------+---------------------------+----------------+---------------------------------+-------------------------+
|0.652027027027027|772          |1184          |31              |702           |638                   |bare_como_curated_ground_v2|1170            |review top pairs before promotion|bad_vehicle_start        |
+-----------------+-------------+--------------+----------------+--------------+----------------------+---------------------------+----------------+----------------

## Output Index


In [10]:
print(f"full candidates: {experiment_dataset_path(EXPERIMENT_SLUG)}")
print(f"accepted candidates: {accepted_path}")
print(f"review artifacts: {experiment_output_path(EXPERIMENT_SLUG)}")
print(f"accepted review sample: {experiment_output_path(EXPERIMENT_SLUG) / 'accepted_review_sample.csv'}")
print(f"promotion summary: {experiment_output_path(EXPERIMENT_SLUG) / 'promotion_summary.csv'}")


full candidates: data/gold/experiments/bare_como_curated_ground_v2/candidates
accepted candidates: data/gold/experiments/bare_como_curated_ground_v2/accepted_candidates
review artifacts: outputs/experiments/bare_como_curated_ground_v2
accepted review sample: outputs/experiments/bare_como_curated_ground_v2/accepted_review_sample.csv
promotion summary: outputs/experiments/bare_como_curated_ground_v2/promotion_summary.csv
